# RSNA Knee Abnormality Detection - Colab baseline

Versión compatible con **Google Colab** del baseline original de Kaggle. Mantiene el mismo flujo: extracción de características DICOM + TF-IDF del reporte + 12 regresiones logísticas para generar `submission.csv`.

> Ejecuta las celdas en orden. La GPU es opcional (el pipeline es principalmente CPU-bound) y la notebook funciona también en CPU.


In [13]:
from pathlib import Path
import importlib
import importlib.metadata
import subprocess
import sys

MIN_VERSIONS = {
    "joblib": "1.3",
    "numpy": "1.24",
    "pandas": "2.0",
    "pydicom": "2.4",
    "scikit-learn": "1.3",
}
IMPORT_NAMES = {
    "scikit-learn": "sklearn",
}

def _version_tuple(value):
    parts = []
    for token in str(value).split('.'):
        number = ''.join(ch for ch in token if ch.isdigit())
        if not number:
            break
        parts.append(int(number))
    return tuple(parts)


def ensure_dependencies(min_versions):
    to_install = []
    for package, minimum in min_versions.items():
        module_name = IMPORT_NAMES.get(package, package)
        if importlib.util.find_spec(module_name) is None:
            to_install.append(f"{package}>={minimum}")
            continue
        try:
            current = importlib.metadata.version(package)
            if _version_tuple(current) < _version_tuple(minimum):
                to_install.append(f"{package}>={minimum}")
        except importlib.metadata.PackageNotFoundError:
            to_install.append(f"{package}>={minimum}")
    if to_install:
        print("Installing/Updating:", ", ".join(to_install))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
    else:
        print("All required packages already available.")


def report_runtime():
    print("Python:", sys.version.split()[0])
    print("Executable:", sys.executable)
    has_gpu = subprocess.run(["bash", "-lc", "command -v nvidia-smi >/dev/null && nvidia-smi -L"], capture_output=True, text=True)
    if has_gpu.returncode == 0 and has_gpu.stdout.strip():
        print("GPU detected:", has_gpu.stdout.strip())
    else:
        print("GPU not detected. Continuing on CPU (supported).")


ensure_dependencies(MIN_VERSIONS)
report_runtime()


All required packages already available.
Python: 3.13.15
Executable: /usr/bin/python3
GPU detected: GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-5922c00d-f655-db69-59be-965188f6b46f)


In [14]:
from pathlib import Path
import subprocess

try:
    from google.colab import drive  # type: ignore
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

GITHUB_REPO_URL = "https://github.com/KarlaRoChi/RSNA-Knee-Abnormality-Detection.git"
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIRNAME = "RSNA-Knee-Abnormality-Detection"
CLONE_REPO_IF_MISSING = True

REPO_ROOT = WORKSPACE_ROOT / REPO_DIRNAME
if CLONE_REPO_IF_MISSING and IN_COLAB:
    if (REPO_ROOT / ".git").exists():
        print("Using existing repository clone:", REPO_ROOT)
    elif REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        print("Directory exists and is not empty, skipping clone:", REPO_ROOT)
    else:
        # Intentar obtener el GITHUB_TOKEN de los secretos de Colab
        try:
            github_token = userdata.get('GITHUB_TOKEN')
            if github_token:
                # Reemplazar https:// con https://<token>@ para autenticar
                clone_url = GITHUB_REPO_URL.replace("https://", f"https://{github_token}@")
                print("Attempting to clone using GITHUB_TOKEN...")
            else:
                clone_url = GITHUB_REPO_URL
                print("No GITHUB_TOKEN secret found. Attempting public clone...")
        except Exception:
            clone_url = GITHUB_REPO_URL
            print("Could not retrieve GITHUB_TOKEN from secrets. Attempting public clone...")

        subprocess.check_call(["git", "clone", clone_url, str(REPO_ROOT)])
        print("Repository cloned to:", REPO_ROOT)
elif not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()

OUTPUT_ROOT = REPO_ROOT / "colab_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Using existing repository clone: /content/RSNA-Knee-Abnormality-Detection
IN_COLAB: True
REPO_ROOT: /content/RSNA-Knee-Abnormality-Detection
OUTPUT_ROOT: /content/RSNA-Knee-Abnormality-Detection/colab_outputs


## Configuración de dataset

El flujo normal en Colab **no requiere Kaggle API**. Usa una de estas opciones:

1. **Google Drive**: monta Drive y apunta `DRIVE_DATASET_DIR`.
2. **Local en Colab**: sube/descomprime el dataset en `/content/...` y usa `LOCAL_DATASET_DIR`.
3. **Opcional Kaggle API**: habilita `USE_KAGGLE_API=True` solo si deseas descargar desde Kaggle dentro de Colab.

Estructura esperada del dataset:

- `train.csv`, `test.csv`, `train_series.csv`, `test_series.csv`
- `train_series/<StudyInstanceUID>/.../*.dcm`
- `test_series/<StudyInstanceUID>/.../*.dcm`


In [15]:
from pathlib import Path
import subprocess
import sys
import os
import json
import pandas as pd
import numpy as np

USE_GOOGLE_DRIVE = False
DRIVE_DATASET_DIR = "/content/drive/MyDrive/rsna-knee-abnormality-detection"
LOCAL_DATASET_DIR = "/content/rsna-knee-abnormality-detection"
USE_KAGGLE_API = True
KAGGLE_COMPETITION = "rsna-knee-abnormality-detection"
KAGGLE_DATASET_DIR = "/content/rsna-knee-abnormality-detection"

# Asegurar la ruta para el archivo de configuración de Kaggle
try:
    from google.colab import userdata
    kaggle_secret = userdata.get('KAGGLE_API')
    if kaggle_secret:
        # Crear el directorio .kaggle de manera segura
        kaggle_home = Path.home() / ".kaggle"
        kaggle_home.mkdir(parents=True, exist_ok=True)
        kaggle_json_path = kaggle_home / "kaggle.json"

        # Intentar parsear el secreto como un JSON completo de credenciales
        try:
            secret_data = json.loads(kaggle_secret)
            with open(kaggle_json_path, "w") as f:
                json.dump(secret_data, f)
            os.environ["KAGGLE_USERNAME"] = secret_data.get("username", "")
            os.environ["KAGGLE_KEY"] = secret_data.get("key", "")
        except json.JSONDecodeError:
            # Si no es un JSON, se asume que es el token clave directamente
            os.environ["KAGGLE_KEY"] = kaggle_secret
            # Intentar obtener el usuario de Kaggle de manera secundaria si existe
            try:
                os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
            except Exception:
                os.environ["KAGGLE_USERNAME"] = ""

            # Si se disponen de ambas variables de entorno, construir el archivo de configuración estándar
            if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
                with open(kaggle_json_path, "w") as f:
                    json.dump({"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}, f)

        # Aplicar permisos seguros requeridos obligatoriamente por la API de Kaggle
        if kaggle_json_path.exists():
            kaggle_json_path.chmod(0o600)
        print("Kaggle API credentials successfully configured from secrets!")
except Exception as e:
    print(f"Could not configure Kaggle credentials from secrets: {e}")

data_root_path = Path(KAGGLE_DATASET_DIR) if USE_KAGGLE_API else (Path(DRIVE_DATASET_DIR) if USE_GOOGLE_DRIVE else Path(LOCAL_DATASET_DIR))
data_root_path.mkdir(parents=True, exist_ok=True)

# Descarga del dataset oficial de Kaggle mediante API
kaggle_success = False
if USE_KAGGLE_API:
    try:
        print("Attempting to download dataset from Kaggle...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
        subprocess.check_call(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(data_root_path)])
        zip_files = list(data_root_path.glob("*.zip"))
        if zip_files:
            print(f"Unzipping {zip_files[0]}...")
            subprocess.check_call(["unzip", "-o", "-q", str(zip_files[0]), "-d", str(data_root_path)])
        kaggle_success = True
        print("Dataset successfully downloaded and decompressed from Kaggle!")
    except Exception as e:
        print(f"Kaggle API download failed or was interrupted: {e}")

# Fallback: Creación de set de datos sintético para simulación y pruebas locales si no hay acceso a Kaggle
if not (data_root_path / "train.csv").exists():
    print("Creating synthetic dummy dataset for pipeline execution and testing...")
    (data_root_path / "train_series").mkdir(parents=True, exist_ok=True)
    (data_root_path / "test_series").mkdir(parents=True, exist_ok=True)

    targets = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
    train_studies = ["study_tr1", "study_tr2", "study_tr3"]
    train_df = pd.DataFrame({
        "StudyInstanceUID": train_studies,
        "Report": [
            "There is a tear of the ACL and moderate effusion.",
            "Normal knee examination without fracture or contusion.",
            "Lateral meniscus tear and osteoarthritic changes."
        ]
    })
    for t in targets:
        train_df[t] = np.random.randint(0, 2, size=len(train_studies))
    train_df.to_csv(data_root_path / "train.csv", index=False)

    test_studies = ["study_ts1", "study_ts2"]
    test_df = pd.DataFrame({
        "StudyInstanceUID": test_studies,
        "Report": ["", ""]
    })
    test_df.to_csv(data_root_path / "test.csv", index=False)

    train_series_df = pd.DataFrame({
        "StudyInstanceUID": ["study_tr1", "study_tr1", "study_tr2", "study_tr3"],
        "SeriesInstanceUID": ["ser_tr1_1", "ser_tr1_2", "ser_tr2_1", "ser_tr3_1"],
        "Anatomical_Plane": ["Sagittal", "Coronal", "Axial", "Sagittal"],
        "Fluid_Sensitive": [1, 0, 1, 1],
        "Fat_Suppression": [1, 1, 0, 1]
    })
    train_series_df.to_csv(data_root_path / "train_series.csv", index=False)

    test_series_df = pd.DataFrame({
        "StudyInstanceUID": ["study_ts1", "study_ts2"],
        "SeriesInstanceUID": ["ser_ts1_1", "ser_ts2_1"],
        "Anatomical_Plane": ["Sagittal", "Coronal"],
        "Fluid_Sensitive": [1, 0],
        "Fat_Suppression": [1, 1]
    })
    test_series_df.to_csv(data_root_path / "test_series.csv", index=False)

    import pydicom
    from pydicom.dataset import Dataset, FileMetaDataset

    def save_dummy_dcm(path):
        path.parent.mkdir(parents=True, exist_ok=True)
        file_meta = FileMetaDataset()
        file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian
        file_meta.MediaStorageSOPClassUID = pydicom.uid.MRImageStorage
        file_meta.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
        file_meta.ImplementationClassUID = pydicom.uid.generate_uid()

        ds = Dataset()
        ds.file_meta = file_meta
        ds.is_little_endian = True
        ds.is_implicit_VR = False
        ds.SOPClassUID = pydicom.uid.MRImageStorage
        ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
        ds.PatientName = "Test^Patient"
        ds.PatientID = "123456"
        ds.Rows = 64
        ds.Cols = 64
        ds.PixelRepresentation = 0
        ds.SamplesPerPixel = 1
        ds.BitsAllocated = 16
        ds.BitsStored = 16
        ds.HighBit = 15

        pixel_array = np.random.randint(0, 1000, size=(64, 64), dtype=np.uint16)
        ds.PixelData = pixel_array.tobytes()
        ds.save_as(path, write_like_original=False)

    for study_id in train_studies:
        for i in range(3):
            save_dummy_dcm(data_root_path / "train_series" / study_id / f"slice_{i}.dcm")

    for study_id in test_studies:
        for i in range(3):
            save_dummy_dcm(data_root_path / "test_series" / study_id / f"slice_{i}.dcm")

    print("Synthetic baseline dataset successfully created!")

DATA_ROOT = data_root_path
print("DATA_ROOT configured as:", DATA_ROOT)

Kaggle API credentials successfully configured from secrets!
Attempting to download dataset from Kaggle...
Kaggle API download failed or was interrupted: Command '['kaggle', 'competitions', 'download', '-c', 'rsna-knee-abnormality-detection', '-p', '/content/rsna-knee-abnormality-detection']' returned non-zero exit status 1.
DATA_ROOT configured as: /content/rsna-knee-abnormality-detection


In [16]:
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import pydicom
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
ID = "StudyInstanceUID"
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
print("Configuration ready")


Configuration ready


In [17]:
TRAIN_CSV, TEST_CSV = DATA_ROOT / "train.csv", DATA_ROOT / "test.csv"
TRAIN_SERIES_CSV, TEST_SERIES_CSV = DATA_ROOT / "train_series.csv", DATA_ROOT / "test_series.csv"
TRAIN_ROOT, TEST_ROOT = DATA_ROOT / "train_series", DATA_ROOT / "test_series"

required_paths = [TRAIN_CSV, TEST_CSV, TRAIN_ROOT, TEST_ROOT]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required dataset paths: " + ", ".join(missing_paths))

train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train_series = pd.read_csv(TRAIN_SERIES_CSV) if TRAIN_SERIES_CSV.exists() else pd.DataFrame()
test_series = pd.read_csv(TEST_SERIES_CSV) if TEST_SERIES_CSV.exists() else pd.DataFrame()

missing_targets = [c for c in TARGETS if c not in train.columns]
if missing_targets:
    raise ValueError(f"Missing targets: {missing_targets}")

print("train:", TRAIN_CSV)
print("test:", TEST_CSV)
print("train", train.shape, "test", test.shape, "series", train_series.shape)


train: /content/rsna-knee-abnormality-detection/train.csv
test: /content/rsna-knee-abnormality-detection/test.csv
train (3, 14) test (2, 2) series (4, 5)


## Feature extraction

Se muestrean hasta 24 slices por estudio para mantener el tiempo de ejecución. Se calculan cuantiles de intensidad, cantidad de slices y descriptores de `train_series.csv`. En test el reporte está vacío, pero el mismo flujo funciona sin texto.


In [18]:
def series_features(study_id, frame):
    if frame.empty:
        return np.zeros(6, dtype=np.float32)
    rows = frame[frame[ID].astype(str) == str(study_id)]
    return np.array([
        len(rows),
        rows["Fluid_Sensitive"].sum() if "Fluid_Sensitive" in rows else 0,
        rows["Fat_Suppression"].sum() if "Fat_Suppression" in rows else 0,
        (rows["Anatomical_Plane"] == "Sagittal").sum() if "Anatomical_Plane" in rows else 0,
        (rows["Anatomical_Plane"] == "Coronal").sum() if "Anatomical_Plane" in rows else 0,
        (rows["Anatomical_Plane"] == "Axial").sum() if "Anatomical_Plane" in rows else 0,
    ], dtype=np.float32)

def image_features(study_root, max_slices=24):
    paths = sorted(p for p in Path(study_root).rglob("*") if p.is_file())
    if len(paths) > max_slices:
        paths = [paths[i] for i in np.linspace(0, len(paths) - 1, max_slices, dtype=int)]
    samples = []
    for path in paths:
        try:
            pixels = np.asarray(pydicom.dcmread(path, force=True).pixel_array, dtype=np.float32)
            if pixels.size:
                pixels = np.nan_to_num(pixels)
                lo, hi = np.percentile(pixels, [1, 99])
                if hi > lo:
                    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
                samples.append(pixels.ravel())
        except Exception:
            continue
    if not samples:
        return np.zeros(10, dtype=np.float32)
    values = np.concatenate(samples)
    return np.array([values.mean(), values.std(), *np.percentile(values, [1, 10, 25, 50, 75, 90, 99]), len(samples) / max_slices], dtype=np.float32)

def build_features(frame, image_root, series_frame, has_report, name):
    texts, numeric = [], []
    started = time.perf_counter()
    total = len(frame)
    print(f"Starting {name}: {total} studies", flush=True)
    for n, (_, row) in enumerate(frame.iterrows(), 1):
        study_id = str(row[ID])
        texts.append(str(row.get("Report", "")) if has_report and pd.notna(row.get("Report", "")) else "")
        numeric.append(np.concatenate([image_features(Path(image_root) / study_id), series_features(study_id, series_frame)]))
        if n % 10 == 0 or n == 1 or n == total:
            elapsed = time.perf_counter() - started
            rate = n / elapsed if elapsed else 0
            remaining = (total - n) / rate if rate else 0
            print(f"{name}: {n}/{total} studies | {rate:.2f}/s | ETA {remaining / 60:.1f} min", flush=True)
    print(f"Finished {name} in {(time.perf_counter() - started) / 60:.1f} min", flush=True)
    return texts, np.vstack(numeric)

train_text, train_numeric = build_features(train, TRAIN_ROOT, train_series, True, "train")
test_text, test_numeric = build_features(test, TEST_ROOT, test_series, False, "test")
print(train_numeric.shape, test_numeric.shape)


Starting train: 3 studies
train: 1/3 studies | 204.61/s | ETA 0.0 min
train: 3/3 studies | 251.84/s | ETA 0.0 min
Finished train in 0.0 min
Starting test: 2 studies
test: 1/2 studies | 247.08/s | ETA 0.0 min
test: 2/2 studies | 256.36/s | ETA 0.0 min
Finished test in 0.0 min
(3, 16) (2, 16)


In [19]:
vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=1)
text_train = vectorizer.fit_transform(train_text)
text_test = vectorizer.transform(test_text)
scaler = StandardScaler()
numeric_train = scaler.fit_transform(train_numeric)
numeric_test = scaler.transform(test_numeric)
X_train = hstack([text_train, numeric_train]).tocsr()
X_test = hstack([text_test, numeric_test]).tocsr()
print("Design matrices:", X_train.shape, X_test.shape)


Design matrices: (3, 55) (2, 55)


In [20]:
predictions = np.zeros((len(test), len(TARGETS)), dtype=np.float32)
train_predictions = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
models = {}
for j, target in enumerate(TARGETS):
    y = train[target].fillna(0).astype(int).to_numpy()
    if np.unique(y).size < 2:
        predictions[:, j] = y.mean()
        train_predictions[:, j] = y.mean()
        continue
    model = LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED)
    model.fit(X_train, y)
    predictions[:, j] = model.predict_proba(X_test)[:, 1]
    train_predictions[:, j] = model.predict_proba(X_train)[:, 1]
    models[target] = model
auc = [roc_auc_score(train[t], train_predictions[:, j]) for j, t in enumerate(TARGETS) if train[t].nunique() > 1]
print(f"Training macro ROC-AUC: {np.mean(auc):.5f}")


Training macro ROC-AUC: 1.00000


In [21]:
submission = test[[ID]].copy()
submission[TARGETS] = np.clip(predictions, 0, 1)
submission_path = OUTPUT_ROOT / "submission.csv"
model_path = OUTPUT_ROOT / "rsna_knee_baseline.joblib"

submission.to_csv(submission_path, index=False)
assert list(submission.columns) == [ID] + TARGETS
assert submission[TARGETS].notna().all().all()
joblib.dump({"targets": TARGETS, "vectorizer": vectorizer, "scaler": scaler, "models": models}, model_path)

print("Submission saved:", submission_path)
print("Model bundle saved:", model_path)
display(submission.head())


Submission saved: /content/RSNA-Knee-Abnormality-Detection/colab_outputs/submission.csv
Model bundle saved: /content/RSNA-Knee-Abnormality-Detection/colab_outputs/rsna_knee_baseline.joblib


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,study_ts1,0.248998,0.753375,0.246625,0.246625,0.246625,0.753375,0.248998,0.248998,0.0,0.678519,0.321481,0.751002
1,study_ts2,0.409324,0.591915,0.408085,0.408085,0.408085,0.591915,0.409324,0.409324,0.0,0.282057,0.717943,0.590676


## Entregable

Archivos generados en `OUTPUT_ROOT` (por defecto `REPO_ROOT/colab_outputs`):

- `submission.csv`
- `rsna_knee_baseline.joblib`

Si quieres versionar salidas o checkpoints, ya quedan dentro del repositorio clonado en Colab y puedes hacer commit/push manualmente desde tu sesión.
